# Silver Layer — Page Views
Clean page view data from Bronze and write standardized Silver table.

## 1. Load bronze table

In [0]:
df = spark.table("shoply_analytics.bronze.page_views")

## 2. Deduplicate page views

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = (
    Window.partitionBy("session_id", "timestamp", "page_url")
          .orderBy(col("session_id"))
)

df_deduped = (
    df.withColumn("rn", row_number().over(window))
      .filter("rn = 1")
      .drop("rn")
)


## 3. Normalize timestamp

In [0]:
from pyspark.sql.functions import to_timestamp, trim

df_clean = (
    df_deduped.withColumn(
        "timestamp",
        to_timestamp(trim(col("timestamp")), "M/d/yy H:mm")
    )
)

## 4. Write to Silver

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable( "shoply_analytics.silver.page_views" )